<a href="https://colab.research.google.com/github/Mainak23/Thirdparty-ai-api-integration/blob/main/UniversalGateway.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [84]:
from google.colab import userdata
userdata.get('token')

'4BOu23lkk5du064DGY2CTra667vLoDQymDjoWtbFGpUttjum1kjGwF6KcEGo5nnN269Ngf5lwJnMOAnBmTbh7LTs5l/vQziQXoUuFCf4fG4U7nEnixJbABhKiwsgMp4KFURUgkYpGyf8xEc='

In [11]:
import openai
import os  # or use colab userdata

# === For Google Colab ===
# from google.colab import userdata

client = openai.OpenAI(
    base_url="https://api.llm7.io/v1",
    api_key=userdata.get('token')  # Get free token from https://token.llm7.io
    # api_key=userdata.get('token')   # Uncomment if using Colab
)

response = client.chat.completions.create(
    model="default",                    # ← Recommended instead of "gpt-4"
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "gsdfwsf"}   # ← Correct format
    ],
    temperature=0.7,
    max_tokens=500
)

print(response.choices[0].message.content)

It seems like you've entered a random string of characters. Could you please provide more context or clarify your question? I'm here to help!


# Introduction

In modern enterprise environments, deploying Large Language Models (LLMs) requires balancing advanced capabilities with rigorous corporate security. Organizations increasingly route all artificial intelligence requests through unified LLM firewalls or secure API gateways. These security layers serve a vital dual purpose: safeguarding against data exfiltration—such as leaking proprietary source code or financial records—and applying real-time context filtering to prevent toxic or non-compliant outputs. While these firewalls provide essential security boundaries, they often alter standard payload parameters, strip metadata, or use non-standard message structures. This project introduces UniChain Gateway, an adaptable, lightweight abstraction client engineered to reconcile custom enterprise infrastructure with modern orchestrators, ensuring uninterrupted development without sacrificing compliance.

# Motivation: Resilient Intelligence via the "Handshake" Protocol
The fundamental challenge of modern enterprise AI engineering lies in the fragmentation of payload standards. When security firewalls intercept data to sanitize context, the complex structural formats required for modern agentic workflows are often broken or completely dropped.

Specifically, native tool-calling functions—where a model pauses text generation to request an external computational step—rely entirely on precise JSON schemas. Standard cloud APIs expect structured nested definitions, whereas hardened internal proxy servers or customized open-source firewalls frequently strip these blocks or strictly require flat, legacy schema arrays. This structural gap breaks the vital two-way handshake loop between the orchestration layer and the cloud engine. If a model cannot reliably yield structured specifications back to the client environment, autonomous agents cannot trigger local code execution, resulting in fractured workflows and lost analytical capabilities.\

                  +---------------------------------------+
                 |  User Input: "What is 15 times 27?"   |
                 +-------------------+-------------------+
                                     |
                                     v
                 +-------------------+-------------------+
                 |      langgraph.create_react_agent     |
                 +-------------------+-------------------+
                                     |
                                     v
                 +-------------------+-------------------+
                 |    UniversalGatewayChatModel._generate|
                 |   - Converts messages to JSON list    |
                 |   - Formats tools into API schemas    |
                 +-------------------+-------------------+
                                     |
                                     | (HTTP POST Request)
                                     v
                 +-------------------+-------------------+
                 |         Remote API Gateway            |
                 |  - Processes text context             |
                 |  - Decides a tool function is needed  |
                 +-------------------+-------------------+
                                     |
                                     | (JSON Response with tool_calls)
                                     v
                 +-------------------+-------------------+
                 |    UniversalGatewayChatModel Parses:  |
                 |  [{'name': 'multiply',                |
                 |    'args': {'a': 15, 'b': 27}}]       |
                 +-------------------+-------------------+
                                     |
                                     v
                 +-------------------+-------------------+
                 |       Local Python Runtime            |
                 |  - Intercepts parsed tool_call structure|
                 |  - Executes local multiply(15, 27)    |
                 |  - Obtains real result: 405           |
                 +-------------------+-------------------+
                                     |
                                     v
                 +-------------------+-------------------+
                 |     ToolMessage Appended to Chain     |
                 |  - Custom model re-invoked with 405   |
                 +-------------------+-------------------+
                                     |
                                     | (HTTP POST Request)
                                     v
                 +-------------------+-------------------+
                 |         Remote API Gateway            |
                 |  - Reads original text + tool result  |
                 |  - Generates plain final text response|
                 +-------------------+-------------------+
                                     |
                                     | (Final Text Response)
                                     v
                 +-------------------+-------------------+
                 |  Final Output: "15 times 27 is 405."  |
                 +---------------------------------------+
                 

In [97]:
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.runnables import RunnableConfig, Runnable
from langchain_core.utils.function_calling import convert_to_openai_tool, convert_to_openai_function
from pydantic import Field, BaseModel
import requests
import json
from typing import List, Any, Iterator, Optional, Dict, Sequence, Union
from langchain_core.language_models.base import LanguageModelInput
from langchain_core.tools import BaseTool

class UniversalGatewayChatModel(BaseChatModel):
    """Robust BaseChatModel engineered to handle both OpenAI-compatible
    and non-OpenAI raw JSON HTTP backends seamlessly."""

    model_name: str = Field(default="default")
    base_url: str = Field(default="https://api.llm7.io/v1")
    api_key: str = Field(default="")
    temperature: float = Field(default=0.7)
    max_tokens: Optional[int] = Field(default=1024)

    # New flags to swap behaviors dynamically depending on the endpoint type
    is_openai_compatible: bool = Field(default=True)
    use_bearer_prefix: bool = Field(default=True)

    def __init__(self, **kwargs: Any):
        super().__init__(**kwargs)

    @property
    def _llm_type(self) -> str:
        return "universal_gateway_chat_model"

    @property
    def supports_tool_calling(self) -> bool:
        return True

    def _convert_messages(self, messages: List[BaseMessage]) -> List[Dict]:
        """Convert LangChain objects to standard serializable payload dictionaries"""
        converted = []
        for message in messages:
            if isinstance(message, SystemMessage):
                converted.append({"role": "system", "content": message.content})
            elif isinstance(message, HumanMessage):
                converted.append({"role": "user", "content": message.content})
            elif isinstance(message, AIMessage):
                msg_dict = {"role": "assistant", "content": message.content or ""}
                # Include incoming tool calls if maintaining a stateful conversation history
                if message.tool_calls and self.is_openai_compatible:
                    msg_dict["tool_calls"] = [
                        {
                            "id": tc.get("id", "call_inst"),
                            "type": "function",
                            "function": {"name": tc["name"], "arguments": json.dumps(tc["args"])}
                        } for tc in message.tool_calls
                    ]
                converted.append(msg_dict)
            else:
                converted.append({"role": "user", "content": message.content})
        return converted

    def _generate(
        self,
        messages: List[BaseMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[Any] = None,
        **kwargs: Any,
    ) -> ChatResult:
        """Synchronous payload execution loop"""

        payload = {
            "model": self.model_name,
            "messages": self._convert_messages(messages),
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }

        # Safely handle tools if passed down from the runtime invoke context
        if "tools" in kwargs and kwargs["tools"]:
            if self.is_openai_compatible:
                payload["tools"] = kwargs["tools"]
                payload["tool_choice"] = kwargs.get("tool_choice", "auto")
            else:
                # Fallback: Extract raw functional structures if backend isn't OpenAI compatible
                payload["functions"] = [
                    t["function"] if "function" in t else t for t in kwargs["tools"]
                ]
                payload["function_call"] = "auto"

        # Dynamically evaluate header constraints
        headers = {"Content-Type": "application/json"}
        if self.api_key:
            if self.use_bearer_prefix and not self.api_key.startswith("Bearer "):
                headers["Authorization"] = f"Bearer {self.api_key}"
            else:
                headers["Authorization"] = self.api_key

        # Route dynamically to standard subpaths or keep raw root endpoints
        endpoint = f"{self.base_url}/chat/completions" if "chat/completions" not in self.base_url else self.base_url

        response = requests.post(endpoint, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()

        # Resilient mapping to extract content irrespective of the container structures
        choice = data.get("choices", [{}])[0] if "choices" in data else data.get("output", {})
        message_data = choice.get("message", choice.get("text", choice))

        # Parsing tool calls dynamically depending on format returned by server
        tool_calls = []
        raw_tool_calls = message_data.get("tool_calls", [])
        raw_function_call = message_data.get("function_call", None)

        if raw_tool_calls:  # OpenAI Standard
            for tc in raw_tool_calls:
                tool_calls.append({
                    "name": tc["function"]["name"],
                    "args": json.loads(tc["function"]["arguments"]) if isinstance(tc["function"]["arguments"], str) else tc["function"]["arguments"],
                    "id": tc.get("id"),
                })
        elif raw_function_call:  # Legacy/Custom API Standard
            tool_calls.append({
                "name": raw_function_call["name"],
                "args": json.loads(raw_function_call["arguments"]) if isinstance(raw_function_call["arguments"], str) else raw_function_call["arguments"],
                "id": "call_fallback"
            })

        ai_message = AIMessage(
            content=message_data.get("content", message_data.get("text", "")) or "",
            tool_calls=tool_calls
        )

        return ChatResult(generations=[ChatGeneration(message=ai_message)])

    def bind_tools(
        self,
        tools: Sequence[Union[Dict[str, Any], BaseTool, type[BaseModel]]],
        **kwargs: Any,
    ) -> Runnable[LanguageModelInput, BaseMessage]:
        """Maps incoming functions using custom formats depending on target engine compatibility"""
        if self.is_openai_compatible:
            formatted_tools = [convert_to_openai_tool(t) for t in tools]
        else:
            # Fallback to standard OpenAI basic function schemas if custom API demands it
            formatted_tools = [convert_to_openai_function(t) for t in tools]

        return self.bind(tools=formatted_tools, **kwargs)

In [98]:
llm_openai = UniversalGatewayChatModel(
    model_name="default",
    base_url="https://api.llm7.io/v1",
    api_key=userdata.get('token'),
    is_openai_compatible=True  # Standard setting
)

# Basic call
response = llm_openai.invoke("Hello, who are you?") # Corrected syntax error
print(response.content)

# With tool calling
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

llm_with_tools = llm_openai.bind_tools([multiply])
result = llm_with_tools.invoke("What is 15 times 27?")
print(result.tool_calls)

Hello! I'm a large language model created by Mistral AI. I'm here to help answer your questions, provide information, and assist with a variety of tasks to the best of my ability. How can I assist you today?
[{'name': 'multiply', 'args': {'a': 15, 'b': 27}, 'id': '66rwpJh8S', 'type': 'tool_call'}]


In [94]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1. First Call: Ask the question
messages = [HumanMessage(content="What is 15 times 27?")]
ai_msg = llm_openai.bind_tools([multiply]).invoke(messages)
messages.append(ai_msg)

# 2. Check if the LLM requested a tool call
if ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        if tool_call["name"] == "multiply":
            # Execute your actual local python function using the extracted args
            tool_output = multiply.invoke(tool_call["args"])

            # Formulate a ToolMessage to feed back into the chat history
            tool_message = ToolMessage(
                content=str(tool_output),
                tool_call_id=tool_call["id"]
            )
            messages.append(tool_message)

    # 3. Second Call: Send the tool result back to the model so it can answer
    final_response = llm_openai.invoke(messages)
    print("Final Text Response:", final_response.content)

Final Text Response: The result of 15 times 27 is **405**.


In [99]:
from langgraph.prebuilt import create_react_agent

# 1. Initialize your custom universal model

# 2. Define your local tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [multiply]

# 3. Create a reactive agent loop that executes the tool automatically
agent_executor = create_react_agent(llm_openai,tools)

# 4. Run the full execution loop
response = agent_executor.invoke({"messages": [("user", "What is 15 times 27?")]})

# LangGraph stores the entire conversational path in the 'messages' key
print("Final Text Response:", response["messages"][-1].content)

/tmp/ipykernel_6588/113130698.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm_openai,tools)


Final Text Response: The result of 15 times 27 is 405.


Ultimately, this project guarantees that enterprise teams do not have to choose between robust data security and advanced agentic capabilities, enabling secure, compliant, and highly functional AI orchestration at scale.